In [6]:
import pandas as pd
import numpy as np
from datetime import datetime

In [7]:
# Generate 5-minute intervals for 2 years
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31, 23, 55)
timestamps = pd.date_range(start=start_date, end=end_date, freq='5min')

In [8]:
# Prepare base DataFrame
df = pd.DataFrame({'Timestamp': timestamps})
df['Date'] = df['Timestamp'].dt.date
df['Hour'] = df['Timestamp'].dt.hour
df['Minute'] = df['Timestamp'].dt.minute
df['TimeInDay'] = df['Hour'] + df['Minute'] / 60
df['DayOfYear'] = df['Timestamp'].dt.dayofyear

In [9]:
# Function to simulate seasonal + diurnal values
def seasonal_5min(dayofyear, timeofday, base, amp_seasonal, amp_diurnal=5, noise=1):
    seasonal = base + amp_seasonal * np.sin(2 * np.pi * dayofyear / 365)
    diurnal = amp_diurnal * np.sin(2 * np.pi * (timeofday - 6) / 24)  # peak around 12PM
    return seasonal + diurnal + np.random.normal(0, noise, len(dayofyear))

In [10]:
# Set seed for reproducibility
np.random.seed(42)


In [ ]:
# Simulate parameters
df['Humidity'] = seasonal_5min(df['DayOfYear'], df['TimeInDay'], 60, 15, 10, 4).clip(30, 100)
df['Temperature'] = seasonal_5min(df['DayOfYear'], df['TimeInDay'], 22, 7, 6, 2).clip(5, 40)
df['Light_Intensity'] = np.where(
    (df['Hour'] >= 6) & (df['Hour'] <= 18),
    seasonal_5min(df['DayOfYear'], df['TimeInDay'], 400, 250, 200, 40),
    np.random.normal(10, 3, len(df))
).clip(0, 1200)
df['Pressure'] = seasonal_5min(df['DayOfYear'], df['TimeInDay'], 1013, 10, 3, 2).clip(980, 1040)

In [ ]:
# Risk label logic
def label_risk(row):
    score = 0
    score += row['Humidity'] > 70
    score += 20 <= row['Temperature'] <= 30
    score += row['Light_Intensity'] < 100
    score += row['Pressure'] < 1000 or row['Pressure'] > 1030

    if score >= 3:
        return "High"
    elif score == 2:
        return "Moderate"
    else:
        return "Low"


In [ ]:
df['Mold_Risk'] = df.apply(label_risk, axis=1)

# Final format
df = df[['Date', 'Timestamp', 'Humidity', 'Temperature', 'Light_Intensity', 'Pressure', 'Mold_Risk']]
df.to_csv("building_mold_risk_5min_1year.csv", index=False)

In [5]:
numeric_cols = data.select_dtypes(include=[np.number]).columns

NameError: name 'df' is not defined